# Red Geográfica de Perfiles de Resistencia — ATLAS

**Definición de la red:**
- **Nodo** = (perfil de resistencia, país)
- **Aristas** = mismo perfil de resistencia en países distintos (Hamming = 0, país diferente)
- **Tamaño del nodo** = frecuencia relativa del perfil en ese país
- **Color del nodo** = número de resistencias (gradiente: claro → oscuro)
- **Posición** = centroide geográfico del país + jitter aleatorio


## 0. Installing geopandas (in my laptop)

In [ ]:
# !pip install --upgrade pip
# pip install --only-binary :all: shapely fiona pyproj rtree geopandas
# I had to install only the binary files because the computer tried to compile libraries from the source code and took forever

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import plotly.graph_objects as go
import urllib.request
import os
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 2. Loading base map (Natural Earth — compatible with GeoPandas ≥ 1.0)

In [ ]:
# How to donwload the GeoJSON file from Natural Earth
# 110m version
# GEOJSON_PATH = "ne_110m_countries.geojson"
# GEOJSON_URL  = (
#     "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
#     "master/geojson/ne_110m_admin_0_countries.geojson"
# )

# # 50m version (includes Hong Kong and Singapore)
# GEOJSON_PATH = "ne_50m_countries.geojson"
# GEOJSON_URL  = (
#     "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
#     "master/geojson/ne_50m_admin_0_countries.geojson"
# )

# if not os.path.exists(GEOJSON_PATH):
#     print("Descargando Natural Earth 110m...")
#     urllib.request.urlretrieve(GEOJSON_URL, GEOJSON_PATH)
#     print("Descargado.")

# world = gpd.read_file(GEOJSON_PATH)
world = gpd.read_file("/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/data/ne_50m_countries.geojson")

# Calculamos centroides (LABEL_X / LABEL_Y ya están ajustados a tierra firme)
# Usamos las columnas LABEL_X / LABEL_Y que Natural Earth incluye explícitamente
# para evitar centroides en el mar (p.ej. Francia, EE.UU.).
centroid_dict = {
    row["NAME"]: (row["LABEL_X"], row["LABEL_Y"])
    for _, row in world.iterrows()
}

print(f"Países en el mapa: {len(centroid_dict)}")
print("Ejemplo:", list(centroid_dict.items())[:3])

## 3. Loading ATLAS data

In [ ]:
# ── Ajusta esta ruta ──────────────────────────────────────────────────────────
CSV_PATH = "/Users/jimenamartinreina/Documents/PabloCatalan/DATABASES/VivliDataChallenge/Vivli_Escherichia_coli.csv"
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)

resistance_columns = [col for col in df.columns if col.endswith("_I")]
df1 = df[resistance_columns].copy()
df1["Country"] = df["Country"]
df1["Year"]    = df["Year"]          # guardamos año como atributo aunque no se use para posición

# Eliminar Colistin (sin variabilidad informativa)
if "Colistin_I" in df1.columns:
    df1.drop("Colistin_I", axis=1, inplace=True)

binary_df = df1.replace({"Resistant": 1, "Susceptible": 0})
resistance_cols = [c for c in binary_df.columns if c not in ("Country", "Year")]

print(f"Antibióticos: {len(resistance_cols)}")
print(f"Filas totales: {len(binary_df)}")
print(f"Países únicos: {binary_df['Country'].nunique()}")

## 4. Mapping country names from ATLAS → Natural Earth

Sometimes they do not share the same name (in ATLAS and in Natural Earth file). With this code we check if we are leaving any country, we can check in the JSON file how it is called and add it in `COUNTRY_NAME_MAP`.

In [ ]:
COUNTRY_NAME_MAP = {
    "United States":              "United States of America",
    "UK":                         "United Kingdom",
    "Czech Republic":             "Czechia",
    "Dominican Republic": "Dominican Rep.",
    "Ivory Coast":        "Côte d'Ivoire",
    "Korea, South":       "South Korea",
    # Add more mappings based on below output
}

def resolve_country(name):
    """Devuelve el nombre Natural Earth para un nombre de país de ATLAS."""
    return COUNTRY_NAME_MAP.get(name, name)

# Countries in ATLAS that are not in the Natural Earth file
atlas_countries = binary_df["Country"].dropna().unique()
missing = [
    c for c in atlas_countries
    if resolve_country(c) not in centroid_dict
]
if missing:
    print(f"⚠️  {len(missing)} countries not found:")
    for m in sorted(missing):
        print(f"   '{m}'")
else:
    print("✅  All ATLAS countries resolved correctly.")

## 5. Nodes: (profile of resistance + country) with relative frequency in that country

In [ ]:
# Total de aislamientos por país (para normalizar)
totals_country = binary_df.groupby("Country").size().rename("total_country")

# Conteo por (perfil + país)
group_cols = resistance_cols + ["Country"]
profile_country_counts = (
    binary_df[group_cols]
    .value_counts()
    .reset_index(name="count")
)
profile_country_counts = profile_country_counts.merge(totals_country, on="Country")
profile_country_counts["freq_relative"] = (
    profile_country_counts["count"] / profile_country_counts["total_country"]
)

print(f"Nodos únicos (perfil, país): {len(profile_country_counts)}")

## 6. Geographic coordenates with jitter

In [ ]:
# Jitter: desplazamiento aleatorio proporcional al número de nodos por país
# para evitar solapamiento. Aumenta JITTER_SCALE si hay muchos nodos por país.
JITTER_SCALE = 2.0  # grados (aprox. 200 km); ajusta al gusto

nodes_data = []
skipped    = []

for _, row in profile_country_counts.iterrows():
    country  = row["Country"]
    resolved = resolve_country(country)

    if resolved not in centroid_dict:
        skipped.append(country)
        continue

    lon0, lat0 = centroid_dict[resolved]
    perfil     = tuple(int(row[c]) for c in resistance_cols)

    # Jitter gaussiano — cada nodo tiene su propio desplazamiento
    jitter_lon = rng.normal(0, JITTER_SCALE)
    jitter_lat = rng.normal(0, JITTER_SCALE)

    nodes_data.append({
        "node_id":       (perfil, country),
        "perfil":        perfil,
        "country":       country,
        "count":         int(row["count"]),
        "total_country": int(row["total_country"]),
        "freq_relative": float(row["freq_relative"]),
        "num_resistant": int(sum(perfil)),
        "lon":           lon0 + jitter_lon,
        "lat":           lat0 + jitter_lat,
        "lon_center":    lon0,
        "lat_center":    lat0,
    })

nodes_df = pd.DataFrame(nodes_data)

if skipped:
    print(f"⚠️  {len(set(skipped))} países omitidos por falta de centroide: {set(skipped)}")
print(f"✅  Nodos con coordenadas: {len(nodes_df)}")

## 7. Edges

In [ ]:
G = nx.Graph()

# Añadir nodos
for _, r in nodes_df.iterrows():
    G.add_node(r["node_id"], **r.to_dict())

# Índice inverso: perfil → lista de node_ids que lo tienen
perfil_to_nodes = defaultdict(list)
for nid in G.nodes():
    perfil_to_nodes[G.nodes[nid]["perfil"]].append(nid)

# Perfiles cosmopolitas (presentes en N países)
for nid in G.nodes():
    G.nodes[nid]["n_countries"] = len(perfil_to_nodes[G.nodes[nid]["perfil"]])
# Update nodes_df with the number of countries for each node
nodes_df["n_countries"] = nodes_df["node_id"].map(
    lambda nid: G.nodes[nid]["n_countries"]
)
print(f"Nodos: {G.number_of_nodes()}")
print(f"Perfiles cosmopolitas (≥5 países): "
      f"{sum(1 for p, ns in perfil_to_nodes.items() if len(ns) >= 5)}")

## 8. Visualization with Plotly in mapamundi

In [ ]:
# ── Parámetros visuales ───────────────────────────────────────────────────────
NODE_SIZE_MIN   = 1    # tamaño mínimo en px
NODE_SIZE_MAX   = 18   # tamaño máximo en px
EDGE_OPACITY    = 0.25 # opacidad de las aristas (0–1)
EDGE_WIDTH      = 0.8
# ─────────────────────────────────────────────────────────────────────────────

# Normalizar tamaño al rango [NODE_SIZE_MIN, NODE_SIZE_MAX]
freq = nodes_df["freq_relative"].values
freq_norm = (freq - freq.min()) / (freq.max() - freq.min() + 1e-12)
node_sizes = NODE_SIZE_MIN + freq_norm * (NODE_SIZE_MAX - NODE_SIZE_MIN)

# ── Trazar nodos ─────────────────────────────────────────────────────────────
hover_texts = [
    (
        f"<b>{r['country']}</b><br>"
        f"Resistencias: {r['num_resistant']}<br>"
        f"Aislamientos: {r['count']} / {r['total_country']} "
        f"({r['freq_relative']*100:.1f}%)<br>"
        f"Países con este perfil: {r['n_countries']}"
    )
    for _, r in nodes_df.iterrows()
]

node_trace = go.Scattergeo(
    lon=nodes_df["lon"],
    lat=nodes_df["lat"],
    mode="markers",
    marker=dict(
        size=node_sizes,
        color=nodes_df["num_resistant"],
        colorscale="YlOrRd",          # claro (pocas) → oscuro (muchas resistencias)
        cmin=0,
        cmax=len(resistance_cols),
        colorbar=dict(
            title="Number of resistances for profile",
            thickness=12,
            len=0.5,
        ),
        line=dict(width=0.4, color="white"),
        opacity=0.85,
    ),
    text=hover_texts,
    hoverinfo="text",
    name="Resistance profiles",
)

# ── Layout ────────────────────────────────────────────────────────────────────
fig = go.Figure([node_trace])

fig.update_geos(
    projection_type="natural earth",
    showland=True,      landcolor="#e8e8e8",
    showocean=True,     oceancolor="#cde5f0",
    showlakes=True,     lakecolor="#cde5f0",
    showcountries=True, countrycolor="white",
    showframe=False,
    bgcolor="#f9f9f9",
)

fig.update_layout(
    title=dict(
        text="Worldwide resistance profiles — <i>E. coli</i> ATLAS",
        x=0.5, xanchor="center",
    ),
    height=700,
    margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(x=0.01, y=0.01),
    paper_bgcolor="#f9f9f9",
)

fig.show()

## 9. Export to GEXF (for Gephi)

In [ ]:
# Gephi espera coordenadas como atributos 'viz' con x/y/z.
# Convertimos lon/lat a x/y para que el mapa sea posicionable en Gephi.
G_export = nx.Graph()

for nid, data in G.nodes(data=True):
    # Convertir node_id (tupla) a string para GEXF
    node_str = f"{data['country']}|{','.join(map(str, data['perfil']))}"
    G_export.add_node(
        node_str,
        country       = data["country"],
        num_resistant = data["num_resistant"],
        count         = data["count"],
        freq_relative = data["freq_relative"],
        n_countries   = data["n_countries"],
        viz={"position": {"x": float(data["lon"]),
                          "y": float(data["lat"]),
                          "z": 0.0}},
    )

# Reconstruir tabla node_id → node_str para las aristas
nid_to_str = {}
for nid, data in G.nodes(data=True):
    nid_to_str[nid] = f"{data['country']}|{','.join(map(str, data['perfil']))}"

for u, v, edata in G.edges(data=True):
    G_export.add_edge(nid_to_str[u], nid_to_str[v], edge_type=edata.get("edge_type", ""))

OUTPUT_PATH = "/Users/jimenamartinreina/Documents/PabloCatalan/REPOSITORY/outputs/networks/geographic_resistance_network_2.gexf"
nx.write_gexf(G_export, OUTPUT_PATH)
print(f"Exportado: {OUTPUT_PATH}")
print(f"  Nodos: {G_export.number_of_nodes()}")
print(f"  Aristas: {G_export.number_of_edges()}")

## 10. Further analysis

### Most cosmopolitan profiles

In [ ]:
# Perfiles presentes en más países
cosmopolitan = (
    pd.DataFrame([
        {"perfil": perfil, "n_countries": len(nids),
         "num_resistant": sum(perfil)}
        for perfil, nids in perfil_to_nodes.items()
    ])
    .sort_values("n_countries", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("Top 20 perfiles más cosmopolitas:")
display(cosmopolitan)

### Resistance tops

In [ ]:
# Tabla resumen por país
summary = []
for country, subset in binary_df.groupby("Country"):
    n_resist = subset[resistance_cols].sum(axis=1)
    n_total = len(subset)
    n_susceptible = (n_resist == 0).sum()
    summary.append({
        "Country": country,
        "n_aislados": n_total,
        "media_resistencias": n_resist.mean(),
        "mediana_resistencias": n_resist.median(),
        "pct_susceptible": n_susceptible / n_total * 100,
    })

country_summary = pd.DataFrame(summary).sort_values("pct_susceptible")

# Top 15 países con MENOS % de nodo susceptible (los más atípicos)
print("Países con menor % de aislados susceptibles a todo:")
display(country_summary.head(15))

# Top 15 con MÁS % susceptible (los que siguen el patrón "normal" más marcadamente)
print("\nPaíses con mayor % de aislados susceptibles a todo:")
display(country_summary.sort_values("pct_susceptible", ascending=False).head(15))

# Top por media de resistencias (de mayor a menor)
print("\nPaíses con mayor media de resistencias:")
display(country_summary.sort_values("media_resistencias", ascending=False).head(15))

# Top por media de resistencias (de menor a mayor)
print("\nPaíses con mayor media de resistencias:")
display(country_summary.sort_values("media_resistencias", ascending=True).head(15))

### Scatter plot of resistance

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from adjustText import adjust_text

fig, ax = plt.subplots(figsize=(14, 10))

norm = mcolors.LogNorm(vmin=country_summary["n_aislados"].min(),
                        vmax=country_summary["n_aislados"].max())

scatter = ax.scatter(
    country_summary["media_resistencias"],
    country_summary["pct_susceptible"],
    s=country_summary["n_aislados"] / 5,
    c=country_summary["n_aislados"],
    cmap="viridis", norm=norm,
    alpha=0.7, edgecolor="white", linewidth=0.5,
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Nº de aislados (escala log)")

ax.set_xlabel("Media de resistencias")
ax.set_ylabel("% de aislados con perfil susceptible a todo")
ax.set_title("Patrón de resistencia por país: media vs. tamaño del nodo susceptible")

# Crear todas las etiquetas, una por país
texts = [
    ax.text(row["media_resistencias"], row["pct_susceptible"], row["Country"], fontsize=7)
    for _, row in country_summary.iterrows()
]

# adjust_text las separa automáticamente y dibuja líneas hacia el punto original
# cuando tiene que desplazarlas
adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5, alpha=0.6),
    expand=(2, 1.5),   # cuánto "espacio personal" exige cada etiqueta
    force_text=(0.3, 0.3),
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from adjustText import adjust_text
from scipy import stats

# ─── Region mapping ────────────────────────────────────────────────────────
# UN M49-based subregions. Adjust country names to match your "Country" column exactly.
REGION_MAP = {
    # --- North America ---
    "United States": "North America", "Canada": "North America",
    "Mexico": "North America",

    # --- Central America & Caribbean ---
    "Costa Rica": "Central America/Caribbean", "Dominican Republic": "Central America/Caribbean",
    "Guatemala": "Central America/Caribbean", "Panama": "Central America/Caribbean",
    "Honduras": "Central America/Caribbean", "El Salvador": "Central America/Caribbean",
    "Jamaica": "Central America/Caribbean", "Puerto Rico": "Central America/Caribbean",

    # --- South America ---
    "Brazil": "South America", "Argentina": "South America",
    "Chile": "South America", "Colombia": "South America",
    "Peru": "South America", "Venezuela": "South America",
    "Bolivia": "South America", "Ecuador": "South America",
    "Uruguay": "South America", "Paraguay": "South America",

    # --- Northern Europe ---
    "United Kingdom": "Northern Europe", "Ireland": "Northern Europe",
    "Sweden": "Northern Europe", "Denmark": "Northern Europe",
    "Norway": "Northern Europe", "Finland": "Northern Europe",     "Latvia":      "Northern Europe",
    "Lithuania":   "Northern Europe",

    # --- Western Europe ---
    "Germany": "Western Europe", "France": "Western Europe",
    "Netherlands": "Western Europe", "Belgium": "Western Europe",
    "Austria": "Western Europe", "Switzerland": "Western Europe",

    # --- Southern Europe ---
    "Spain": "Southern Europe", "Italy": "Southern Europe",
    "Portugal": "Southern Europe", "Greece": "Southern Europe", "Croatia":     "Southern Europe",
    "Slovenia":    "Southern Europe",

    # --- Eastern Europe ---
    "Poland": "Eastern Europe", "Russia": "Eastern Europe",
    "Czech Republic": "Eastern Europe", "Hungary": "Eastern Europe",
    "Romania": "Eastern Europe", "Ukraine": "Eastern Europe", "Bulgaria": "Eastern Europe",

    # --- North Africa ---
    "Egypt": "North Africa", "Morocco": "North Africa",
    "Tunisia": "North Africa", "Algeria": "North Africa",

    # --- Sub-Saharan Africa ---
    "Nigeria": "Sub-Saharan Africa", "Kenya": "Sub-Saharan Africa",
    "Malawi": "Sub-Saharan Africa", "South Africa": "Sub-Saharan Africa",
    "Tanzania": "Sub-Saharan Africa", "Ivory Coast": "Sub-Saharan Africa",
    "Ghana": "Sub-Saharan Africa", "Uganda": "Sub-Saharan Africa", "Cameroon":    "Sub-Saharan Africa",

    # --- East Asia ---
    "China": "East Asia", "Japan": "East Asia",
    "South Korea": "East Asia", "Hong Kong": "East Asia",
    "Taiwan": "East Asia", "Korea, South": "East Asia",

    # --- South Asia ---
    "India": "South Asia",

    # --- Southeast Asia ---
    "Thailand": "Southeast Asia", "Vietnam": "Southeast Asia",
    "Philippines": "Southeast Asia", "Malaysia": "Southeast Asia",
    "Singapore": "Southeast Asia", "Indonesia": "Southeast Asia",

    # --- Western Asia / Middle East ---
    "Turkey": "Western Asia", "Israel": "Western Asia",
    "Saudi Arabia": "Western Asia", "Lebanon": "Western Asia",
    "Syria": "Western Asia", "Iran": "Western Asia",     "Jordan":      "Western Asia",
    "Kuwait":      "Western Asia",
    "Qatar":       "Western Asia",

    # --- Oceania ---
    "Australia": "Oceania", "New Zealand": "Oceania",
}

# Diagnostic: which countries in your dataset are NOT in the dictionary?
countries_without_region = set(country_summary["Country"]) - set(REGION_MAP.keys())
if countries_without_region:
    print(f"⚠️  {len(countries_without_region)} countries without a region assigned — add them to the dictionary:")
    for c in sorted(countries_without_region):
        print(f"   '{c}'")

In [ ]:
# ─── Plot with color by region ──────────────────────────────────────────────
country_summary["region"] = country_summary["Country"].map(REGION_MAP)

# Palette: one color per region, grouping shades by continent so that
# Northern/Southern/Eastern Europe look like a "family" of blues, etc.
region_colors = {
    "North America":              "#1f77b4",
    "Central America/Caribbean":  "#6baed6",
    "South America":              "#9ecae1",
    "Northern Europe":            "#2ca02c",
    "Western Europe":             "#74c476",
    "Southern Europe":            "#a1d99b",
    "Eastern Europe":             "#c7e9c0",
    "North Africa":               "#ff7f0e",
    "Sub-Saharan Africa":         "#fdae6b",
    "East Asia":                  "#d62728",
    "South Asia":                 "#ff9896",
    "Southeast Asia":             "#e377c2",
    "Western Asia":               "#f7b6d2",
    "Oceania":                    "#9467bd",
}

fig, ax = plt.subplots(figsize=(14, 10))

for region, color in region_colors.items():
    subset = country_summary[country_summary["region"] == region]
    if subset.empty:
        continue
    ax.scatter(
        subset["media_resistencias"], subset["pct_susceptible"],
        s=subset["n_aislados"] / 5,
        color=color, label=region,
        alpha=0.75, edgecolor="white", linewidth=0.5,
    )

ax.set_xlabel("Mean number of resistances")
ax.set_ylabel("% of isolates with fully susceptible profile")
ax.set_title("Resistance pattern by country and geographic region")
ax.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

texts = [
    ax.text(row["media_resistencias"], row["pct_susceptible"], row["Country"], fontsize=7)
    for _, row in country_summary.iterrows()
]
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.4, alpha=0.5),
            expand=(1.2, 1.5))

plt.tight_layout()
plt.show()

### K Means on the scatter plot

In [ ]:
country_summary

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# data
col_media_antibioticos = 'media_resistencias'
col_tamano_susceptible = 'pct_susceptible'
features = [col_media_antibioticos, col_tamano_susceptible]
df_kmeans = country_summary[features].dropna().copy()

# Scaling data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

# define and inizialite KMeans
# random_state to get the same
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')

# train model
df_kmeans['cluster'] = kmeans.fit_predict(X_scaled)

# add cluster column into country_summary
country_summary['cluster'] = df_kmeans['cluster']

# Represent
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=country_summary, 
    x=col_media_antibioticos, 
    y=col_tamano_susceptible, 
    hue='cluster', 
    palette='tab10',
    s=100,
    alpha=0.8
)
plt.title(f'KMeans in {n_clusters} clusters', fontsize=14, fontweight='bold')
plt.xlabel('Mean number of resistances', fontsize=12)
plt.ylabel('% of isolates with fully susceptible profile', fontsize=12)
plt.legend(title='Cluster')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─── Estilo general "paper" ──────────────────────────────────────────────
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
})

country_summary["region"] = country_summary["Country"].map(REGION_MAP)

region_colors = {
    "North America":              "#1f77b4",
    "Central America/Caribbean":  "#6baed6",
    "South America":              "#9ecae1",
    "Northern Europe":            "#2ca02c",
    "Western Europe":             "#74c476",
    "Southern Europe":            "#a1d99b",
    "Eastern Europe":             "#c7e9c0",
    "North Africa":               "#ff7f0e",
    "Sub-Saharan Africa":         "#fdae6b",
    "East Asia":                  "#d62728",
    "South Asia":                 "#ff9896",
    "Southeast Asia":             "#e377c2",
    "Western Asia":               "#f7b6d2",
    "Oceania":                    "#9467bd",
}

fig, ax = plt.subplots(figsize=(14, 10))

for region, color in region_colors.items():
    subset = country_summary[country_summary["region"] == region]
    if subset.empty:
        continue
    ax.scatter(
        subset["media_resistencias"], subset["pct_susceptible"],
        s=subset["n_aislados"] / 5,
        color=color, label=region,
        alpha=0.8, edgecolor="white", linewidth=0.6,
        zorder=3,
    )

# ─── Líneas divisorias entre clusters ───────────────────────────────────
# Ajusta estos puntos a ojo hasta que separen bien tus datos
ax.plot([3.7, 5.6], [20.5, 37], color="black", lw=1.1, zorder=2)
ax.plot([6.6, 8.5], [1.5, 20.5], color="black", lw=1.1, zorder=2)

# ─── Etiquetas de los tres grupos ───────────────────────────────────────
ax.text(3.0, 49,
        "Europe + North and Center\nAmerica + Australia + Japan\n+ New Zealand",
        fontsize=10, va="top")

ax.text(6.2, 33,
        "Asia + Russia + Ukraine +\nMiddle East + Morocco +\nSouth Africa + South\nAmerica",
        fontsize=10, va="top")

ax.text(7.9, 12,
        "Subsaharan Africa, India,\nMexico, Dominican Republic",
        fontsize=10, va="top")

# ─── Ejes y título ───────────────────────────────────────────────────────
ax.set_xlabel("Mean number of resistances")
ax.set_ylabel("% of isolates with fully susceptible profile")
ax.set_title("Resistance pattern by country and geographic region", fontweight="bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.25, linestyle="--", linewidth=0.5, zorder=0)

# Leyenda: solo un marcador por región (sin escalar con el tamaño)
handles, labels = ax.get_legend_handles_labels()
leg = ax.legend(handles, labels, title="Region",
                bbox_to_anchor=(1.02, 1), loc="upper left",
                fontsize=9, title_fontsize=10, frameon=True,
                markerscale=1.5, handletextpad=0.8)
leg.get_frame().set_edgecolor("#cccccc")

# ─── Etiquetas de país con adjustText ───────────────────────────────────
texts = [
    ax.text(row["media_resistencias"], row["pct_susceptible"], row["Country"], fontsize=7.5)
    for _, row in country_summary.iterrows()
]
adjust_text(texts, ax=ax,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.4, alpha=0.6),
            expand=(1.2, 1.5))

plt.tight_layout()

# ─── Exportar en alta resolución para el paper ──────────────────────────
# plt.savefig("resistance_pattern.png", dpi=300, bbox_inches="tight")
# plt.savefig("resistance_pattern.pdf", bbox_inches="tight") 
plt.show()

## 11. Statistics

In [ ]:
# ─── ANOVA: does region explain variability in the susceptible-profile % ? ──
groups = [g["pct_susceptible"].values for _, g in country_summary.groupby("region") if len(g) > 1]
f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA pct_susceptible by region: F={f_stat:.2f}, p={p_value:.4f}")

In [ ]:
h_stat, p_value_kw = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H={h_stat:.2f}, p={p_value_kw:.4f}")

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(
    endog=country_summary["pct_susceptible"],
    groups=country_summary["region"],
    alpha=0.05
)
print(tukey)

## 12. Speciality column

In [ ]:
countries_to_plot = ["Nigeria", "China", "India", "Thailand", "Malawi", "Costa Rica"]

df_temp = df.copy()
df_temp["n_resist"] = binary_df[resistance_cols].sum(axis=1)

for country in countries_to_plot:
    print(f"\n{country}:")
    print(
        df_temp[df_temp["Country"] == country]
        .groupby("Speciality")["n_resist"]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
    )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

countries_to_plot = df["Country"].unique().tolist()

# Construir tabla país x especialidad con la media de n_resist
heatmap_data = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Speciality"])["n_resist"]
    .mean()
    .unstack("Speciality")
)

# Reordenar filas según el orden que quieras (opcional)
heatmap_data = heatmap_data.reindex(countries_to_plot)

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y especialidad clínica")
ax.set_xlabel("Especialidad")
ax.set_ylabel("País")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_counts = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Speciality"])["n_resist"]
    .count()
    .unstack("Speciality")
    .reindex(countries_to_plot)
)

mask = heatmap_counts < 10  # oculta celdas con menos de 10 aislados

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data, mask=mask,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y especialidad (celdas con n<10 ocultas)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 13. Gender

In [ ]:
countries_to_plot = df["Country"].unique().tolist()

# Construir tabla país x especialidad con la media de n_resist
heatmap_data = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Gender"])["n_resist"]
    .mean()
    .unstack("Gender")
)

# Reordenar filas según el orden que quieras (opcional)
heatmap_data = heatmap_data.reindex(countries_to_plot)

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y género")
ax.set_xlabel("Género")
ax.set_ylabel("País")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_counts = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Gender"])["n_resist"]
    .count()
    .unstack("Gender")
    .reindex(countries_to_plot)
)

mask = heatmap_counts < 10  # oculta celdas con menos de 10 aislados

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data, mask=mask,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y género (celdas con n<10 ocultas)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 14. Source

In [ ]:
countries_to_plot = df["Country"].unique().tolist()

# Construir tabla país x especialidad con la media de n_resist
heatmap_data = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Source"])["n_resist"]
    .mean()
    .unstack("Source")
)

# Reordenar filas según el orden que quieras (opcional)
heatmap_data = heatmap_data.reindex(countries_to_plot)

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y fuente")
ax.set_xlabel("Fuente")
ax.set_ylabel("País")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_counts = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Source"])["n_resist"]
    .count()
    .unstack("Source")
    .reindex(countries_to_plot)
)

mask = heatmap_counts < 10  # oculta celdas con menos de 10 aislados

fig, ax = plt.subplots(figsize=(20, 20))
sns.heatmap(
    heatmap_data, mask=mask,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y fuente (celdas con n<10 ocultas)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 15. Age Group

In [ ]:
countries_to_plot = df["Country"].unique().tolist()

# Construir tabla país x especialidad con la media de n_resist
heatmap_data = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Age Group"])["n_resist"]
    .mean()
    .unstack("Age Group")
)

# Reordenar filas según el orden que quieras (opcional)
heatmap_data = heatmap_data.reindex(countries_to_plot)

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y grupo de edad")
ax.set_xlabel("Grupo de edad")
ax.set_ylabel("País")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_counts = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Age Group"])["n_resist"]
    .count()
    .unstack("Age Group")
    .reindex(countries_to_plot)
)

mask = heatmap_counts < 10  # oculta celdas con menos de 10 aislados

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data, mask=mask,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y grupo de edad (celdas con n<10 ocultas)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 16. Year

In [ ]:
# Construir tabla país x especialidad con la media de n_resist
heatmap_data = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Year"])["n_resist"]
    .mean()
    .unstack("Year")
)

# Reordenar filas según el orden que quieras (opcional)
heatmap_data = heatmap_data.reindex(countries_to_plot)

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y año")
ax.set_xlabel("Año")
ax.set_ylabel("País")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_counts = (
    df_temp[df_temp["Country"].isin(countries_to_plot)]
    .groupby(["Country", "Year"])["n_resist"]
    .count()
    .unstack("Year")
    .reindex(countries_to_plot)
)

mask = heatmap_counts < 10  # oculta celdas con menos de 10 aislados

fig, ax = plt.subplots(figsize=(13, 20))
sns.heatmap(
    heatmap_data, mask=mask,
    annot=True, fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Media de resistencias"},
    ax=ax,
)
ax.set_title("Media de resistencias por país y año (celdas con n<10 ocultas)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

## 17. ANOVA

In [ ]:
from scipy import stats

metadata_cols = ["Speciality", "Source", "Age Group", "Gender", "Year"]
# Ajusta los nombres exactos de columna según tu df

results = []

for country in df_temp["Country"].unique():
    subset = df_temp[df_temp["Country"] == country]
    total_var = subset["n_resist"].var()
    if total_var == 0 or len(subset) < 10:
        continue

    row = {"Country": country, "n": len(subset), "total_variance": total_var}

    for col in metadata_cols:
        # Elimina NaN para ese metadato
        clean = subset[["n_resist", col]].dropna()
        groups = [g["n_resist"].values for _, g in clean.groupby(col) if len(g) >= 3]
        if len(groups) < 2:
            row[f"eta2_{col}"] = None
            continue

        f_stat, p_val = stats.f_oneway(*groups)

        # η² = varianza entre grupos / varianza total
        grand_mean = clean["n_resist"].mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
        ss_total   = ((clean["n_resist"] - grand_mean)**2).sum()
        eta2 = ss_between / ss_total if ss_total > 0 else None

        row[f"eta2_{col}"] = round(eta2, 3) if eta2 is not None else None
        row[f"p_{col}"]    = round(p_val, 4)

    results.append(row)

variance_explained = pd.DataFrame(results).sort_values("total_variance", ascending=False)
display(variance_explained)

In [ ]:
eta2_cols = [c for c in variance_explained.columns if c.startswith("eta2_")]
p_cols    = [c for c in variance_explained.columns if c.startswith("p_")]

# Para cada variable de metadatos, muestra los casos con efecto relevante Y significativo
THRESHOLD_ETA2 = 0.06   # efecto mediano según convención
THRESHOLD_P    = 0.05   # significación estadística

for eta_col, p_col in zip(eta2_cols, p_cols):
    variable = eta_col.replace("eta2_", "")
    mask = (
        variance_explained[eta_col] >= THRESHOLD_ETA2
    ) & (
        variance_explained[p_col] < THRESHOLD_P
    )
    subset = variance_explained[mask][["Country", "n", eta_col, p_col]]
    if not subset.empty:
        print(f"\n── {variable} (η²≥{THRESHOLD_ETA2}, p<{THRESHOLD_P}) ──")
        print(subset.sort_values(eta_col, ascending=False).to_string(index=False))
    else:
        print(f"\n── {variable}: ningún país supera el umbral ──")